In [5]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

/Users/macuser/projects/ufc_prediction/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [37]:
def make_soup(url: str) -> BeautifulSoup:
    source_code = requests.get(url, allow_redirects=False)
    plain_text = source_code.text.encode("ascii", "replace")
    return BeautifulSoup(plain_text, "html.parser")

def get_all_event_details(soup_obj):

    all_event_links = []

    for link in soup_obj.find_all("td", {"class": "b-statistics__table-col"}):
                for href in link.find_all("a"):
                    event_link = href.get("href")
                    all_event_links.append(event_link)

    return all_event_links

def get_all_fight_details(event_soup_obj):
      
    fight_details_links = []

    for fight_detail in event_soup_obj.find_all("tr", {"class": "b-fight-details__table-row b-fight-details__table-row__hover js-fight-details-click"}):
        href = fight_detail.get("data-link")
        fight_details_links.append(href)

    return fight_details_links


In [41]:
all_events_url="http://ufcstats.com/statistics/events/completed?page=all"

test_soup = make_soup(all_events_url)

all_event_links = get_all_event_details(test_soup)

test_event = all_event_links[1]

test_event_soup = make_soup(test_event)

fight_links = get_all_fight_details(test_event_soup)
fight_links[:5]

['http://ufcstats.com/fight-details/85e94a6c071fd9fa',
 'http://ufcstats.com/fight-details/9a0ecee6e8ca173f',
 'http://ufcstats.com/fight-details/6102e7a6dcbc62f0',
 'http://ufcstats.com/fight-details/f9f5071b993a5f4b',
 'http://ufcstats.com/fight-details/e32dc26b11d1f81d']

In [50]:
fight_soup = make_soup(fight_links[0])


In [111]:
tables = fight_soup.findAll("tbody")
test_table = tables[0]


/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_22862/459022420.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  tables = fight_soup.findAll("tbody")


In [ ]:
row = test_table.find("tr")
stats = []
for data in row.find_all("td"):
                if stats == "":
                    stats = data.text
                else:
                    stats = stats + "," + data.text
stats.replace("\n\n", "").replace("  ", "").replace("\n", ",").replace(", ", ",").replace(" ,", ",")

'Israel Adesanya,Joe Pyfer,0,0,42 of 75,36 of 70,56%,51%,58 of 92,52 of 90,0 of 2,2 of 8,0%,25%,0,0,0,0,0:00,2:44'

In [159]:
row = test_table.find("tr")
raw_stats = []
for data in row.find_all("td"):
                dat_clean = data.text.replace("\n\n", "").replace("      ","").replace("\n    \n","")
                if raw_stats == []:
                    raw_stats = [dat_clean]
                else:
                    raw_stats.append(dat_clean)
# stats.replace("\n\n", "")replace("\n\n", "").replace("  ", "").replace("\n\n", "").replace("\n", ",").replace(", ", ",").replace(" ,", ",")
raw_stats

['Israel Adesanya \nJoe Pyfer ',
 '0\n    0',
 '42 of 75\n    36 of 70',
 '56%\n    51%',
 '58 of 92\n    52 of 90',
 '0 of 2\n    2 of 8',
 '0%\n    25%',
 '0\n    0',
 '0\n    0',
 '0:00\n    2:44']

In [110]:
def convert_to_named_stats(raw_stats):
    named_stats = {}
    named_stats['fighters'] = raw_stats[0].split('\n')
    named_stats['knockdowns'] = raw_stats[1].split('\n    ')
    named_stats['significant strikes'] = raw_stats[2].split('\n    ')
    named_stats['significant strike percentage'] = raw_stats[3].split('\n    ')
    named_stats['total strikes'] = raw_stats[4].split('\n    ')
    named_stats['takedowns'] = raw_stats[5].split('\n    ')
    named_stats['takedown percentage'] = raw_stats[6].split('\n    ')
    named_stats['submission attempts'] = raw_stats[7].split('\n    ')
    named_stats['reversals'] = raw_stats[8].split('\n    ')
    named_stats['control'] = raw_stats[9].split('\n    ')

    return named_stats

convert_to_named_stats(raw_stats)

{'fighters': ['Israel Adesanya ', 'Joe Pyfer '],
 'knockdowns': ['0', '0'],
 'significant strikes': ['42 of 75', '36 of 70'],
 'significant strike percentage': ['56%', '51%'],
 'total strikes': ['58 of 92', '52 of 90'],
 'takedowns': ['0 of 2', '2 of 8'],
 'takedown percentage': ['0%', '25%'],
 'submission attempts': ['0', '0'],
 'reversals': ['0', '0'],
 'control': ['0:00', '2:44']}

In [221]:
totals_table_structure = {0: ['fighters', '\n'],
                          1: ['knockdowns', '\n    '],
                          2: ['significant strikes', '\n    '],
                          3: ['significant strike percentage', '\n    '],
                          4: ['total strike', '\n    '],
                          5: ['takedowns', '\n    '],
                          6: ['takedown percentage', '\n    '],
                          7: ['submission attempts', '\n    '],
                          8: ['reversals', '\n    '],
                          9: ['control', '\n    ']}

for i in totals_table_structure:
    print(i, totals_table_structure[i][0], raw_stats[i].split(totals_table_structure[i][1]))

0 fighters ['Israel Adesanya ', 'Joe Pyfer ']
1 knockdowns ['0', '0']
2 significant strikes ['42 of 75', '36 of 70']
3 significant strike percentage ['56%', '51%']
4 total strike ['58 of 92', '52 of 90']
5 takedowns ['0 of 2', '2 of 8']
6 takedown percentage ['0%', '25%']
7 submission attempts ['0', '0']
8 reversals ['0', '0']
9 control ['0:00', '2:44']


In [141]:
tables = fight_soup.findAll("tbody")
totals_table = tables[0]
totals_per_round_table = tables[1]
significant_strikes_table = tables[2]
significant_strikes_per_round_table = tables[3]


/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_22862/4181895494.py:1: DeprecationWarning: Call to deprecated method findAll. (Replaced by find_all) -- Deprecated since version 4.0.0.
  tables = fight_soup.findAll("tbody")


In [167]:
## Use find_all to get a result set, then seperate
row = totals_table.find("tr")
raw_stats_test = []
for data in row.find_all("td"):
                dat_clean = data.text.replace("\n\n", "").replace("      ","").replace("\n    \n","")
                if raw_stats_test == []:
                    raw_stats_test = [dat_clean]
                else:
                    raw_stats_test.append(dat_clean)
# stats.replace("\n\n", "")replace("\n\n", "").replace("  ", "").replace("\n\n", "").replace("\n", ",").replace(", ", ",").replace(" ,", ",")
raw_stats_test

['Israel Adesanya \nJoe Pyfer ',
 '0\n    0',
 '42 of 75\n    36 of 70',
 '56%\n    51%',
 '58 of 92\n    52 of 90',
 '0 of 2\n    2 of 8',
 '0%\n    25%',
 '0\n    0',
 '0\n    0',
 '0:00\n    2:44']

In [170]:
type(totals_per_round_table.find('tr'))
# type(totals_per_round_table.find_all('tr'))

bs4.element.Tag

In [153]:
len(totals_per_round_table.find_all('tr'))

2

In [ ]:
round_dict = {
    'round' : '',
    'fighter_a' : '',
    'fighter_b' : '',
    'knockdowns_a': '',
    'knockdowns_b': '',
    'sig_str_a':'',
    'sig_str_b':'',
    'sig_str_pct_a':'',
    'sig_str_pct_b':'',
    'total_str_a':'',
    'total_str_b':'',

}

In [ ]:
## get all of the table data and initialize the data dictionary
totals_dat = totals_table.find_all('td')
test_total_dict = {}

for i in range(len(totals_dat)):
    ## remove new lines and tabs for easier formatting
    table_row_data_clean = totals_dat[i].text.replace("\n\n", "").replace("      ","").replace("\n    \n","")

    ## fill in the dictioanry for the corresponding structure
    test_total_dict[totals_table_structure[i][0]] = table_row_data_clean.split(totals_table_structure[i][1])

print(test_total_dict)

In [259]:
# totals_table
def extract_data_from_table(table, table_structure):

    ## get all of the table data and initialize the data dictionary
    table_data = table.find_all('td')
    table_dict = {}

    for i in range(len(table_data)):
        ## remove new lines and tabs for easier formatting
        table_data_clean = table_data[i].text.replace("\n\n", "").replace("      ","").replace("\n    \n","")

        ## fill in the dictioanry for the corresponding structure
        table_dict[table_structure[i][0]] = table_data_clean.split(table_structure[i][1])

    return table_dict

extract_data_from_table(totals_table,totals_table_structure)


{'fighters': ['Israel Adesanya ', 'Joe Pyfer '],
 'knockdowns': ['0', '0'],
 'significant strikes': ['42 of 75', '36 of 70'],
 'significant strike percentage': ['56%', '51%'],
 'total strike': ['58 of 92', '52 of 90'],
 'takedowns': ['0 of 2', '2 of 8'],
 'takedown percentage': ['0%', '25%'],
 'submission attempts': ['0', '0'],
 'reversals': ['0', '0'],
 'control': ['0:00', '2:44']}

In [247]:
def parse_per_round_table(per_round_table, per_round_table_structure):

    ## Save all table headers and table rows:
    headers_and_rows = per_round_table.find_all(['th','tr'])

    ## Initialize the stats_per_round_dict and round unumber object
    stats_per_round_dict = {}
    round = None

    for i in headers_and_rows:

        ## if the soup_item is a table header, set our round variable to it
        if i.name == 'th':
            round_num = i.text.replace("\n              ", "").replace("\n            ","")
            # print(round_num)

        ## if the soup_item is a table row, parse the table row data and assign it to the current round
        if i.name == 'tr':
            round_row_data = i.find_all('td')

            ind_round_dict = {}

            for i in range(len(round_row_data)):

                ## remove new lines and tabs for easier formatting
                round_row_data_clean = round_row_data[i].text.replace("\n\n", "").replace("      ","").replace("\n    \n","")

                ## fill in the individual_round_dictioanry for the corresponding structure
                ind_round_dict[per_round_table_structure[i][0]] = round_row_data_clean.split(per_round_table_structure[i][1])

            stats_per_round_dict[round_num] = ind_round_dict

    return stats_per_round_dict

parse_per_round_table(totals_per_round_table,totals_table_structure)

{'Round 1': {'fighters': ['Israel Adesanya ', 'Joe Pyfer '],
  'knockdowns': ['0', '0'],
  'significant strikes': ['24 of 42', '9 of 23'],
  'significant strike percentage': ['57%', '39%'],
  'total strike': ['37 of 56', '9 of 23'],
  'takedowns': ['0 of 0', '1 of 4'],
  'takedown percentage': ['---', '25%'],
  'submission attempts': ['0', '0'],
  'reversals': ['0', '0'],
  'control': ['0:00', '1:01']},
 'Round 2': {'fighters': ['Israel Adesanya ', 'Joe Pyfer '],
  'knockdowns': ['0', '0'],
  'significant strikes': ['18 of 33', '27 of 47'],
  'significant strike percentage': ['54%', '57%'],
  'total strike': ['21 of 36', '43 of 67'],
  'takedowns': ['0 of 2', '1 of 4'],
  'takedown percentage': ['0%', '25%'],
  'submission attempts': ['0', '0'],
  'reversals': ['0', '0'],
  'control': ['0:00', '1:43']}}

In [ ]:

## Save all table headers and table rows:
headers_and_rows = totals_per_round_table.find_all(['th','tr'])

## Initialize the stats_per_round_dict and round unumber object
stats_per_round_dict = {}
round = None

for i in headers_and_rows:

    ## if the soup_item is a table header, set our round variable to it
    if i.name == 'th':
        round_num = i.text.replace("\n              ", "").replace("\n            ","")
        # print(round_num)

    ## if the soup_item is a table row, parse the table row data and assign it to the current round
    if i.name == 'tr':
        round_row_data = i.find_all('td')

        ind_round_dict = {}

        for i in range(len(round_row_data)):

            ## remove new lines and tabs for easier formatting
            round_row_data_clean = round_row_data[i].text.replace("\n\n", "").replace("      ","").replace("\n    \n","")

            ## fill in the individual_round_dictioanry for the corresponding structure
            ind_round_dict[totals_table_structure[i][0]] = round_row_data_clean.split(totals_table_structure[i][1])

        stats_per_round_dict[round_num] = ind_round_dict
print(stats_per_round_dict.keys())
print(stats_per_round_dict)




dict_keys(['Round 1', 'Round 2'])
{'Round 1': {'fighters': ['Israel Adesanya ', 'Joe Pyfer '], 'knockdowns': ['0', '0'], 'significant strikes': ['24 of 42', '9 of 23'], 'significant strike percentage': ['57%', '39%'], 'total strike': ['37 of 56', '9 of 23'], 'takedowns': ['0 of 0', '1 of 4'], 'takedown percentage': ['---', '25%'], 'submission attempts': ['0', '0'], 'reversals': ['0', '0'], 'control': ['0:00', '1:01']}, 'Round 2': {'fighters': ['Israel Adesanya ', 'Joe Pyfer '], 'knockdowns': ['0', '0'], 'significant strikes': ['18 of 33', '27 of 47'], 'significant strike percentage': ['54%', '57%'], 'total strike': ['21 of 36', '43 of 67'], 'takedowns': ['0 of 2', '1 of 4'], 'takedown percentage': ['0%', '25%'], 'submission attempts': ['0', '0'], 'reversals': ['0', '0'], 'control': ['0:00', '1:43']}}


In [ ]:
#function: 
#   seperate fight data into different sections  -- DONE
#       for each fight section:
#           extract the correct amount of tables (1 or 2)
#   for each extracted table:
#       get the information using a pre-made dictioanry. 
#       Append the resulting row to a csv

def seperate_fight_into_sections(fight_soup):

    ## seperate fight data into different sections 
    totals = fight_soup[0]
    totals_per_round = fight_soup[1]
    sig_str = fight_soup[2]
    sig_str_per_round = fight_soup[3]

    ## totals and sig_strikes_table are only 1 table, totals_per_round and sig_str_per_round have multiple sections that need to be condensed

    ## combine per_round tables into a single table

    ## parse totals_per_round table
    totals_per_round_data = parse_per_round_table(totals_per_round, totals_table_structure)



    return totals_per_round_data
